# УПРОЩЕННЫЙ CLTV - БЕЗ CHURN, БЫСТРОЕ ОБУЧЕНИЕ
# Два сегмента: small и large_and_middle

In [ ]:
# %% ИМПОРТЫ И НАСТРОЙКИ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import warnings
from collections import defaultdict, deque
import logging
import pickle
import json
from pathlib import Path

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("Импорты загружены")

In [ ]:
# %% КОНФИГУРАЦИЯ
class Config:
    # Пути к Parquet файлам
    DATA_DIR = Path("data")
    TRAIN_PATH = DATA_DIR / "train_data.parquet"
    PROD_PATH = DATA_DIR / "prod_data.parquet"
    
    MODEL_DIR = Path("models_simple")
    MODEL_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    FORECAST_START = "2025-10-31"
    HORIZON_MONTHS = 6
    VALIDATION_CUTOFF = "2025-03-31"
    MIN_SAMPLES_PER_SEGMENT = 1000
    
    CATEGORICAL_FEATURES = ['QUALITY_CODE', 'SUBJECT_KIND_ID', 'EC_SECTOR_ID']
    BASE_FEATURES = [
        'MARGIN', 'MARGIN_LAG1', 'MARGIN_LAG2', 'MARGIN_LAG3',
        'MARGIN_AVG_1M_LAG', 'MARGIN_AVG_2M_LAG', 'MARGIN_AVG_3M_LAG',
        'MARGIN_AVG_6M_LAG', 'MARGIN_AVG_12M_LAG', 'MARGIN_STDDEV_12M_LAG',
        'MARGIN_GROWTH_RATE_3M', 'MONTH_OF_YEAR', 'QUARTER_OF_YEAR', 'TENURE_MONTHS'
    ]
    
    # Новый маппинг сегментов (изменено!)
    SEGMENT_MAPPING = {
        '1026': 'small',              # MICRO
        '1027': 'small',              # SMALL
        '1022': 'large_and_middle',   # MIDDLE
        '1023': 'large_and_middle',   # LARGE
        '1040': 'large_and_middle',   # → large_and_middle (изменено!)
        '1028': 'large_and_middle',   # На всякий случай
    }
    
    # Быстрые параметры CatBoost (без Optuna)
    CATBOOST_PARAMS = {
        'iterations': 500,           # Быстрое обучение
        'depth': 4,
        'learning_rate': 0.05,
        'l2_leaf_reg': 3,
        'random_seed': 42,
        'loss_function': 'RMSE',
        'verbose': 100,
        'early_stopping_rounds': 50
    }
    
    @classmethod
    def ensure_directories(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        (cls.MODEL_DIR / cls.MODEL_VERSION).mkdir(parents=True, exist_ok=True)

Config.ensure_directories()
print(f"Версия: {Config.MODEL_VERSION}")
print(f"\nМаппинг сегментов:")
for old_seg, new_seg in sorted(Config.SEGMENT_MAPPING.items()):
    print(f"  {old_seg} → {new_seg}")

In [ ]:
# %% ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def read_parquet(path):
    """Чтение Parquet файлов"""
    if not os.path.exists(path):
        logger.warning(f"Файл не найден: {path}")
        return pd.DataFrame()
    return pd.read_parquet(path)

def fix_categorical_features(df, cat_features):
    df_fixed = df.copy()
    for col in cat_features:
        if col in df_fixed.columns:
            df_fixed[col] = df_fixed[col].fillna('UNKNOWN').astype(str)
            df_fixed[col] = df_fixed[col].str.replace('.0', '', regex=False)
    return df_fixed

def stabilize_target(y):
    """Log-трансформация таргета"""
    return np.sign(y) * np.log1p(np.abs(y))

def inverse_stabilize_target(y_stable):
    """Обратная трансформация"""
    return np.sign(y_stable) * (np.exp(np.abs(y_stable)) - 1)

print("Функции загружены")

In [ ]:
# %% ЗАГРУЗКА ДАННЫХ ИЗ PARQUET
logger.info("Загрузка данных из Parquet...")

train = read_parquet(Config.TRAIN_PATH)
prod = read_parquet(Config.PROD_PATH)

print(f"\nОбучающая: {len(train):,} записей, {train['CLIENT_ID'].nunique():,} клиентов")
print(f"Продакшн: {len(prod):,} записей")

if train.empty or prod.empty:
    print("\n⚠️  ВНИМАНИЕ: Файлы не загружены!")
    print("Запустите сначала notebook 'data_loader.ipynb'")
else:
    print("\n✅ Данные успешно загружены")

In [ ]:
# %% ОБЪЕДИНЕНИЕ СЕГМЕНТОВ
logger.info("Объединение сегментов...")

print("\n" + "="*70)
print("ОРИГИНАЛЬНЫЕ СЕГМЕНТЫ")
print("="*70)
print(train['SEGMENT_ID'].value_counts().sort_index())

# Преобразование к строке и применение маппинга
train['SEGMENT_ID'] = train['SEGMENT_ID'].astype(str).map(Config.SEGMENT_MAPPING)
prod['SEGMENT_ID'] = prod['SEGMENT_ID'].astype(str).map(Config.SEGMENT_MAPPING)

# Обработка неизвестных сегментов
train['SEGMENT_ID'] = train['SEGMENT_ID'].fillna('large_and_middle')
prod['SEGMENT_ID'] = prod['SEGMENT_ID'].fillna('large_and_middle')

print("\n" + "="*70)
print("НОВЫЕ ОБЪЕДИНЕННЫЕ СЕГМЕНТЫ (2 сегмента)")
print("="*70)
print(train['SEGMENT_ID'].value_counts().sort_index())

print("\nДетальное распределение в TRAIN:")
for segment in sorted(train['SEGMENT_ID'].unique()):
    seg_data = train[train['SEGMENT_ID'] == segment]
    count = len(seg_data)
    clients = seg_data['CLIENT_ID'].nunique()
    pct = 100 * count / len(train)
    avg_margin = seg_data['TARGET_NEXT_MARGIN'].mean()
    print(f"  {segment:20s}: {count:>10,} записей ({pct:>5.1f}%), {clients:>8,} клиентов, avg margin: {avg_margin:>12,.0f}")

print("\nДетальное распределение в PROD:")
for segment in sorted(prod['SEGMENT_ID'].unique()):
    seg_data = prod[prod['SEGMENT_ID'] == segment]
    count = len(seg_data)
    pct = 100 * count / len(prod)
    avg_margin = seg_data['MARGIN'].mean() if 'MARGIN' in prod.columns else 0
    print(f"  {segment:20s}: {count:>10,} записей ({pct:>5.1f}%), avg margin: {avg_margin:>12,.0f}")

In [ ]:
# %% ПРЕДОБРАБОТКА
all_categorical = Config.CATEGORICAL_FEATURES + ['SEGMENT_ID']
train_fixed = fix_categorical_features(train, all_categorical)
prod_fixed = fix_categorical_features(prod, all_categorical)

ALL_FEATURES = ['SEGMENT_ID'] + Config.BASE_FEATURES + Config.CATEGORICAL_FEATURES
available_features = [f for f in ALL_FEATURES if f in train_fixed.columns]

numeric_features = [f for f in available_features if f not in all_categorical]
for col in numeric_features:
    train_fixed[col] = train_fixed[col].fillna(0.0)
    if col in prod_fixed.columns:
        prod_fixed[col] = prod_fixed[col].fillna(0.0)

print(f"Фичи готовы: {len(available_features)}")
print(f"Числовые фичи: {len(numeric_features)}")
print(f"Категориальные фичи: {len(Config.CATEGORICAL_FEATURES)}")
print(f"Сегменты: {sorted(train_fixed['SEGMENT_ID'].unique())}")

In [ ]:
# %% КЛАСС ДЛЯ ОБУЧЕНИЯ МОДЕЛЕЙ (УПРОЩЕННЫЙ)
class SimpleCLTV:
    """Упрощенная версия - без Optuna, без churn"""
    
    def __init__(self, catboost_params):
        self.models = {}
        self.segment_stats = {}
        self.feature_importance = {}
        self.catboost_params = catboost_params
        
    def train_segment(self, segment_id, X_train, y_train, X_val, y_val, cat_features=None):
        """Обучение модели для одного сегмента"""
        print(f"\nОбучение сегмента: {segment_id}")
        print(f"  Train samples: {len(X_train):,}")
        print(f"  Val samples: {len(X_val):,}")
        
        # Определение индексов категориальных признаков
        cat_indices = []
        if cat_features:
            features_list = X_train.columns.tolist()
            for cat_feat in cat_features:
                if cat_feat in features_list:
                    cat_indices.append(features_list.index(cat_feat))
        
        # Создание Pool
        train_pool = Pool(X_train, y_train, cat_features=cat_indices)
        val_pool = Pool(X_val, y_val, cat_features=cat_indices)
        
        # Обучение
        model = CatBoostRegressor(**self.catboost_params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)
        
        # Метрики
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        
        metrics = {
            'train_r2': r2_score(y_train, train_pred),
            'val_r2': r2_score(y_val, val_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, train_pred)),
            'val_rmse': np.sqrt(mean_squared_error(y_val, val_pred)),
            'train_mae': mean_absolute_error(y_train, train_pred),
            'val_mae': mean_absolute_error(y_val, val_pred),
            'train_samples': len(X_train),
            'val_samples': len(X_val)
        }
        
        # Feature importance
        importance_df = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\n  Результаты:")
        print(f"    R² train: {metrics['train_r2']:.4f}, R² val: {metrics['val_r2']:.4f}")
        print(f"    RMSE train: {metrics['train_rmse']:.2f}, RMSE val: {metrics['val_rmse']:.2f}")
        print(f"    MAE train: {metrics['train_mae']:.2f}, MAE val: {metrics['val_mae']:.2f}")
        
        print(f"\n  Топ-10 важных фичей:")
        for idx, row in importance_df.head(10).iterrows():
            print(f"    {row['feature']:30s}: {row['importance']:.2f}")
        
        self.models[segment_id] = model
        self.segment_stats[segment_id] = metrics
        self.feature_importance[segment_id] = importance_df
        
        return model, metrics
    
    def predict(self, segment_id, X):
        """Предсказание для сегмента"""
        if segment_id not in self.models:
            raise ValueError(f"Нет модели для сегмента {segment_id}")
        
        X_pred = X.drop('SEGMENT_ID', axis=1) if 'SEGMENT_ID' in X.columns else X
        return self.models[segment_id].predict(X_pred)

print("Класс SimpleCLTV создан")

In [ ]:
# %% ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
print("\n" + "="*70)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ")
print("="*70)

# Стабилизация таргета
train_fixed['target_stable'] = stabilize_target(train_fixed['TARGET_NEXT_MARGIN'])

# Разделение на train/val по времени
train_mask = pd.to_datetime(train_fixed['MONTH_END']) <= pd.to_datetime(Config.VALIDATION_CUTOFF)
val_mask = ~train_mask

print(f"\nОбщее разделение:")
print(f"  Train: {train_mask.sum():,} записей")
print(f"  Val: {val_mask.sum():,} записей")

# Подготовка данных по сегментам
segment_data = {}

for segment_id in sorted(train_fixed['SEGMENT_ID'].unique()):
    seg_mask = train_fixed['SEGMENT_ID'] == segment_id
    
    seg_train_mask = seg_mask & train_mask
    seg_val_mask = seg_mask & val_mask
    
    features_for_segment = [f for f in available_features if f != 'SEGMENT_ID']
    
    X_train = train_fixed[seg_train_mask][features_for_segment]
    y_train = train_fixed[seg_train_mask]['target_stable']
    X_val = train_fixed[seg_val_mask][features_for_segment]
    y_val = train_fixed[seg_val_mask]['target_stable']
    
    segment_data[segment_id] = {
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val
    }
    
    print(f"\nСегмент {segment_id}:")
    print(f"  Train: {len(X_train):,} записей")
    print(f"  Val: {len(X_val):,} записей")

print("\nДанные подготовлены")

In [ ]:
# %% ОБУЧЕНИЕ МОДЕЛЕЙ
print("\n" + "="*70)
print("НАЧИНАЕМ ОБУЧЕНИЕ МОДЕЛЕЙ")
print("="*70)

cltv_model = SimpleCLTV(catboost_params=Config.CATBOOST_PARAMS)

for segment_id in sorted(segment_data.keys()):
    print(f"\n{'='*70}")
    print(f"СЕГМЕНТ: {segment_id.upper()}")
    print(f"{'='*70}")
    
    data = segment_data[segment_id]
    
    model, metrics = cltv_model.train_segment(
        segment_id=segment_id,
        X_train=data['X_train'],
        y_train=data['y_train'],
        X_val=data['X_val'],
        y_val=data['y_val'],
        cat_features=Config.CATEGORICAL_FEATURES
    )

print("\n" + "="*70)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("="*70)

In [ ]:
# %% СВОДКА РЕЗУЛЬТАТОВ
print("\n" + "="*70)
print("СВОДКА ПО МОДЕЛЯМ")
print("="*70)

summary_data = []
for segment_id, stats in cltv_model.segment_stats.items():
    summary_data.append({
        'Сегмент': segment_id,
        'Train_R2': f"{stats['train_r2']:.4f}",
        'Val_R2': f"{stats['val_r2']:.4f}",
        'Train_RMSE': f"{stats['train_rmse']:.1f}",
        'Val_RMSE': f"{stats['val_rmse']:.1f}",
        'Train_MAE': f"{stats['train_mae']:.1f}",
        'Val_MAE': f"{stats['val_mae']:.1f}",
        'Train_Samples': f"{stats['train_samples']:,}",
        'Val_Samples': f"{stats['val_samples']:,}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Средние метрики
avg_train_r2 = np.mean([stats['train_r2'] for stats in cltv_model.segment_stats.values()])
avg_val_r2 = np.mean([stats['val_r2'] for stats in cltv_model.segment_stats.values()])
avg_val_rmse = np.mean([stats['val_rmse'] for stats in cltv_model.segment_stats.values()])
avg_val_mae = np.mean([stats['val_mae'] for stats in cltv_model.segment_stats.values()])

print(f"\n{'='*70}")
print("СРЕДНИЕ МЕТРИКИ:")
print(f"  Train R²: {avg_train_r2:.4f}")
print(f"  Val R²: {avg_val_r2:.4f}")
print(f"  Val RMSE: {avg_val_rmse:.1f}")
print(f"  Val MAE: {avg_val_mae:.1f}")
print(f"{'='*70}")

In [ ]:
# %% АНАЛИЗ ПРЕДСКАЗАНИЙ НА ВАЛИДАЦИИ
print("\n" + "="*70)
print("АНАЛИЗ ПРЕДСКАЗАНИЙ НА ВАЛИДАЦИИ")
print("="*70)

for segment_id in sorted(cltv_model.models.keys()):
    print(f"\nСегмент: {segment_id}")
    
    data = segment_data[segment_id]
    y_val_pred_stable = cltv_model.predict(segment_id, data['X_val'])
    
    # Обратная трансформация
    y_val_true = inverse_stabilize_target(data['y_val'])
    y_val_pred = inverse_stabilize_target(y_val_pred_stable)
    
    # Статистика
    print(f"\n  Истинные значения (TARGET_NEXT_MARGIN):")
    print(f"    Mean: {y_val_true.mean():,.0f}")
    print(f"    Median: {y_val_true.median():,.0f}")
    print(f"    Std: {y_val_true.std():,.0f}")
    print(f"    Min: {y_val_true.min():,.0f}")
    print(f"    Max: {y_val_true.max():,.0f}")
    
    print(f"\n  Предсказанные значения:")
    print(f"    Mean: {y_val_pred.mean():,.0f}")
    print(f"    Median: {np.median(y_val_pred):,.0f}")
    print(f"    Std: {y_val_pred.std():,.0f}")
    print(f"    Min: {y_val_pred.min():,.0f}")
    print(f"    Max: {y_val_pred.max():,.0f}")
    
    # Метрики на оригинальной шкале
    r2_orig = r2_score(y_val_true, y_val_pred)
    rmse_orig = np.sqrt(mean_squared_error(y_val_true, y_val_pred))
    mae_orig = mean_absolute_error(y_val_true, y_val_pred)
    
    print(f"\n  Метрики на оригинальной шкале:")
    print(f"    R²: {r2_orig:.4f}")
    print(f"    RMSE: {rmse_orig:,.0f}")
    print(f"    MAE: {mae_orig:,.0f}")

In [ ]:
# %% СОХРАНЕНИЕ МОДЕЛЕЙ
print("\n" + "="*70)
print("СОХРАНЕНИЕ МОДЕЛЕЙ")
print("="*70)

save_path = Config.MODEL_DIR / Config.MODEL_VERSION

# Сохранение CatBoost моделей
for segment_id, model in cltv_model.models.items():
    model_file = save_path / f"model_{segment_id}.cbm"
    model.save_model(str(model_file))
    print(f"  ✓ Сохранена модель: {model_file}")

# Сохранение метрик
summary_df.to_csv(save_path / "metrics_summary.csv", index=False)
print(f"  ✓ Сохранены метрики: {save_path / 'metrics_summary.csv'}")

# Сохранение feature importance
for segment_id, importance_df in cltv_model.feature_importance.items():
    importance_file = save_path / f"feature_importance_{segment_id}.csv"
    importance_df.to_csv(importance_file, index=False)
    print(f"  ✓ Сохранена важность фичей: {importance_file}")

# Метаданные
metadata = {
    'version': Config.MODEL_VERSION,
    'train_date': datetime.now().isoformat(),
    'segments': list(cltv_model.models.keys()),
    'features': available_features,
    'segment_mapping': Config.SEGMENT_MAPPING,
    'catboost_params': Config.CATBOOST_PARAMS,
    'metrics': {seg: {k: float(v) if isinstance(v, (int, float, np.number)) else v 
                     for k, v in stats.items()} 
               for seg, stats in cltv_model.segment_stats.items()}
}

with open(save_path / "metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"  ✓ Сохранены метаданные: {save_path / 'metadata.json'}")

# Сохранение объекта модели
with open(save_path / "model_object.pkl", 'wb') as f:
    pickle.dump(cltv_model, f)
print(f"  ✓ Сохранен объект модели: {save_path / 'model_object.pkl'}")

print(f"\n{'='*70}")
print(f"✅ ВСЕ МОДЕЛИ СОХРАНЕНЫ В: {save_path}")
print(f"{'='*70}")

In [ ]:
# %% ПРОСТОЕ ПРОГНОЗИРОВАНИЕ (БЕЗ CHURN)
print("\n" + "="*70)
print("ПРОСТОЕ ПРОГНОЗИРОВАНИЕ НА ПРОДАКШН ДАННЫХ")
print("="*70)

# Подготовка фичей для каждого сегмента
predictions_list = []

for segment_id in sorted(cltv_model.models.keys()):
    seg_prod = prod_fixed[prod_fixed['SEGMENT_ID'] == segment_id].copy()
    
    if len(seg_prod) == 0:
        print(f"\nСегмент {segment_id}: нет данных в prod")
        continue
    
    print(f"\nСегмент {segment_id}: {len(seg_prod):,} клиентов")
    
    # Фичи без SEGMENT_ID
    features_for_pred = [f for f in available_features if f != 'SEGMENT_ID']
    X_prod = seg_prod[features_for_pred]
    
    # Предсказание
    y_pred_stable = cltv_model.predict(segment_id, X_prod)
    y_pred = inverse_stabilize_target(y_pred_stable)
    
    # Добавление в результаты
    seg_prod['PREDICTED_MARGIN'] = y_pred
    seg_prod['PREDICTED_MARGIN_STABLE'] = y_pred_stable
    
    predictions_list.append(seg_prod[['CLIENT_ID', 'SEGMENT_ID', 'MARGIN', 'PREDICTED_MARGIN']])
    
    print(f"  Статистика предсказаний:")
    print(f"    Mean: {y_pred.mean():,.0f}")
    print(f"    Median: {np.median(y_pred):,.0f}")
    print(f"    Std: {y_pred.std():,.0f}")

# Объединение всех предсказаний
all_predictions = pd.concat(predictions_list, ignore_index=True)

print(f"\n{'='*70}")
print(f"Всего предсказаний: {len(all_predictions):,}")
print(f"{'='*70}")

# Сохранение предсказаний
predictions_file = save_path / "predictions_prod.csv"
all_predictions.to_csv(predictions_file, index=False)
print(f"\n✅ Предсказания сохранены: {predictions_file}")

# Статистика по сегментам
print(f"\nСтатистика предсказаний по сегментам:")
segment_stats = all_predictions.groupby('SEGMENT_ID')['PREDICTED_MARGIN'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(0)
print(segment_stats)

In [ ]:
# %% ВИЗУАЛИЗАЦИЯ ВАЖНОСТИ ФИЧЕЙ
print("\n" + "="*70)
print("ВИЗУАЛИЗАЦИЯ ВАЖНОСТИ ФИЧЕЙ")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, segment_id in enumerate(sorted(cltv_model.models.keys())):
    importance_df = cltv_model.feature_importance[segment_id].head(15)
    
    ax = axes[idx]
    ax.barh(range(len(importance_df)), importance_df['importance'])
    ax.set_yticks(range(len(importance_df)))
    ax.set_yticklabels(importance_df['feature'])
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importance - {segment_id}')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(save_path / 'feature_importance.png', dpi=100, bbox_inches='tight')
print(f"\n✅ График сохранен: {save_path / 'feature_importance.png'}")
plt.show()

In [ ]:
# %% ИТОГИ
print("\n" + "="*70)
print("ИТОГОВАЯ ИНФОРМАЦИЯ")
print("="*70)

print(f"\n📊 РЕЗУЛЬТАТЫ ОБУЧЕНИЯ:")
print(f"  ✓ Обучено моделей: {len(cltv_model.models)}")
print(f"  ✓ Сегменты: {', '.join(sorted(cltv_model.models.keys()))}")
print(f"  ✓ Средний Val R²: {avg_val_r2:.4f}")
print(f"  ✓ Средний Val MAE: {avg_val_mae:,.0f}")

print(f"\n💾 СОХРАНЕННЫЕ ФАЙЛЫ:")
print(f"  ✓ Директория: {save_path}")
print(f"  ✓ Модели CatBoost: {len(cltv_model.models)} файлов")
print(f"  ✓ Метрики: metrics_summary.csv")
print(f"  ✓ Предсказания: predictions_prod.csv")
print(f"  ✓ Метаданные: metadata.json")

print(f"\n📈 ПРЕДСКАЗАНИЯ:")
print(f"  ✓ Всего клиентов: {len(all_predictions):,}")
for segment_id in sorted(all_predictions['SEGMENT_ID'].unique()):
    seg_count = len(all_predictions[all_predictions['SEGMENT_ID'] == segment_id])
    seg_mean = all_predictions[all_predictions['SEGMENT_ID'] == segment_id]['PREDICTED_MARGIN'].mean()
    print(f"  ✓ {segment_id}: {seg_count:,} клиентов, средний прогноз: {seg_mean:,.0f}")

print(f"\n{'='*70}")
print("✅ ОБУЧЕНИЕ И ПРОГНОЗИРОВАНИЕ ЗАВЕРШЕНО УСПЕШНО!")
print(f"{'='*70}")